In [1]:
import scanpy as sc
import os
from datetime import datetime

os.environ['TZ'] = 'Europe/Berlin'
import time
time.tzset()

def checkpoint(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

In [2]:
checkpoint("Loading data")
adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/10_integration_GBM_HIHA/gbmap_hiha_concat_harmony.h5ad")
checkpoint(f"Data loaded: {adata.n_obs} cells, {adata.n_vars} genes")

[2026-09-17 15:22:51] Loading data
[2026-09-17 15:22:58] Data loaded: 208945 cells, 22586 genes


In [3]:
OUTPUT = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/10_integration_GBM_HIHA"

In [4]:
TARGET_CELLTYPES = [
    "Macrophages SPP1",
    "Macro. and mono. prolif.",
    "TAMs C1QC",
    "TAMs proinflammatory",
    "Monocytes",
]


In [5]:
adata_sub = adata[adata.obs["decoupler_level1_celltype"].isin(TARGET_CELLTYPES)].copy()
print("Subset shape:", adata_sub.shape)
print(adata_sub.obs["decoupler_level1_celltype"].value_counts())

Subset shape: (43358, 22586)
decoupler_level1_celltype
Monocytes                   15528
Macro. and mono. prolif.     9067
TAMs proinflammatory         6911
TAMs C1QC                    6852
Macrophages SPP1             5000
Name: count, dtype: int64


In [6]:
print(f"X.min()={adata_sub.X.min():.3f}  X.max()={adata_sub.X.max():.3f}  "
      f"log1p flag={adata_sub.uns.get('log1p')}")
assert adata_sub.X.min() >= 0, (
    "adata_sub.X has negative values - NMF requires non-negative input. "
    "Rebuild from raw counts -> normalize_total -> log1p before proceeding."
)

X.min()=0.000  X.max()=9.039  log1p flag=None


In [7]:
import anndata as ad
ad.settings.allow_write_nullable_strings = True


adata_sub.write(os.path.join(OUTPUT, "10_subset_macro.h5ad"))


In [10]:
adata_sub.obs["donor_id"].value_counts()

donor_id
Patient_4    2486
rGBM-02      1390
Patient_2    1344
Patient_1    1258
R5            816
             ... 
PJ018           1
PJ053           1
E37             1
SF11215         1
SF11979         1
Name: count, Length: 281, dtype: int64